# Task C – SOC Automation System Prototype

This notebook stitches the phishing classifier into a lightweight SOC automation flow. It ingests recent emails, enriches them with IOC extraction, derives analyst-facing risk levels and recommendations, stores the results in a triage dataframe, and produces situational awareness visualisations.

## 1. Initialise project context

Ensure the Task A artifacts are available and add the repository modules to the Python path.

In [ ]:
from pathlib import Path
import random
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the project root so that 'src/' is available.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
if not (ARTIFACT_DIR / "best_model.joblib").exists():
    raise FileNotFoundError("Missing best_model.joblib. Execute Task A before running this notebook.")

random.seed(42)
print(f"Artifacts ready at {ARTIFACT_DIR}")

## 2. Helper utilities for SOC triage

We reuse the Task B pipeline for scoring and IOC enrichment, then compute derived fields (severity, priority score, recommendations) that downstream dashboards expect.

In [ ]:
from datetime import datetime, timedeltafrom typing import Dict, Listimport numpy as npimport pandas as pdfrom src.task_b_soc.pipeline import classify_text, reload_modelfrom src.task_b_soc.enrich import extract_iocs, summarize_iocsreload_model()RISK_TO_SEVERITY = {    "high": "Critical",    "elevated": "High",    "moderate": "Medium",    "low": "Low",}def triage_message(subject: str, body: str, source: str, received_ts: datetime) -> Dict:    combined = f"{subject}{body}".strip()    model_output = classify_text(combined)    iocs = extract_iocs(combined)    summary = summarize_iocs(iocs)    risk_level = model_output.get("risk_level", "low")    severity = RISK_TO_SEVERITY.get(risk_level, "Low")    probability = float(model_output.get("score", 0.0))    priority_score = probability * 100 + summary.get("total", 0) * 5    return {        "timestamp": received_ts,        "source": source,        "subject": subject,        "phishing_prob": probability,        "label": model_output.get("label"),        "risk_level": risk_level,        "severity": severity,        "priority_score": round(priority_score, 2),        "confidence": model_output.get("confidence"),        "ioc_summary": summary,        "iocs": iocs,        "recommendations": model_output.get("recommendations", []),        "rationale": model_output.get("explanations", {}).get("rationale"),    }

## 3. Build a triage queue from recent emails

For demonstration purposes we sample messages from the processed Task A dataset and assign pseudo timestamps over the last 24 hours.

In [ ]:
from datetime import datetime

PROCESSED_TEST_PATH = PROJECT_ROOT / "data" / "processed" / "test.csv"
if not PROCESSED_TEST_PATH.exists():
    raise FileNotFoundError("Missing processed test split. Run Task A preprocessing first.")

raw_df = pd.read_csv(PROCESSED_TEST_PATH)

sample_size = min(50, len(raw_df))
window_hours = 24
base_time = datetime.utcnow()

records: List[Dict] = []
for row in raw_df.sample(n=sample_size, random_state=42).itertuples(index=False):
    subject = getattr(row, "subject", "") if hasattr(row, "subject") else getattr(row, "text", "")[:80]
    body = getattr(row, "text", "")
    source = getattr(row, "source", "unknown")
    delta = timedelta(hours=random.uniform(0, window_hours))
    timestamp = base_time - delta
    triaged = triage_message(subject=subject, body=body, source=source, received_ts=timestamp)
    records.append(triaged)

alerts_df = pd.DataFrame(records).sort_values("timestamp", ascending=False).reset_index(drop=True)
alerts_df.head()

## 4. Prioritise alerts and compute queue statistics

In [ ]:
alerts_df["priority_bucket"] = pd.cut(
    alerts_df["priority_score"],
    bins=[-np.inf, 40, 60, 80, np.inf],
    labels=["Low", "Medium", "High", "Critical"],
)

queue_stats = {
    "total_alerts": len(alerts_df),
    "critical_or_high": int((alerts_df["priority_bucket"].isin(["High", "Critical"])).sum()),
    "avg_priority": float(alerts_df["priority_score"].mean()),
    "ioc_hit_rate": float((alerts_df["ioc_summary"].apply(lambda x: x.get("total", 0) > 0)).mean()),
}

queue_stats

## 5. Visualise SOC trends

The plots are saved under `reports/` for inclusion in the thesis or presentations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

REPORT_FIG_DIR = PROJECT_ROOT / "reports"
REPORT_FIG_DIR.mkdir(parents=True, exist_ok=True)

viz_df = alerts_df.copy()
viz_df["timestamp"] = pd.to_datetime(viz_df["timestamp"])
viz_df.sort_values("timestamp", inplace=True)

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

viz_df.set_index("timestamp").resample("2H").size().plot(ax=axes[0, 0], marker="o", color="#d62728")
axes[0, 0].set_title("Alerts over time")
axes[0, 0].set_xlabel("Timestamp")
axes[0, 0].set_ylabel("Count")

severity_counts = viz_df["severity"].value_counts().reindex(["Critical", "High", "Medium", "Low"], fill_value=0)
severity_counts.plot(kind="bar", ax=axes[0, 1], color="#1f77b4")
axes[0, 1].set_title("Severity distribution")
axes[0, 1].set_xlabel("Severity")
axes[0, 1].set_ylabel("Alerts")

primary_sources = viz_df["source"].value_counts().head(10)
primary_sources.plot(kind="barh", ax=axes[1, 0], color="#2ca02c")
axes[1, 0].invert_yaxis()
axes[1, 0].set_title("Top sources")
axes[1, 0].set_xlabel("Alerts")

scatter = axes[1, 1].scatter(
    viz_df["phishing_prob"],
    viz_df["ioc_summary"].apply(lambda x: x.get("total", 0)),
    c=viz_df["priority_score"],
    cmap="viridis",
    alpha=0.7,
)
axes[1, 1].set_title("Probability vs IOC volume")
axes[1, 1].set_xlabel("Model probability")
axes[1, 1].set_ylabel("Total IOCs")
fig.colorbar(scatter, ax=axes[1, 1]).set_label("Priority score")

plt.tight_layout()
figure_path = REPORT_FIG_DIR / "soc_dashboard.png"
plt.savefig(figure_path, dpi=200)
plt.show()

print(f"Dashboard saved to: {figure_path}")

## 6. Inspect a high-priority alert

Review the IOC context, rationale, and recommended actions that the SOC platform would surface to an analyst.

In [ ]:
high_priority = alerts_df.sort_values("priority_score", ascending=False).iloc[0]
{
    "subject": high_priority["subject"],
    "severity": high_priority["severity"],
    "probability": high_priority["phishing_prob"],
    "risk_level": high_priority["risk_level"],
    "rationale": high_priority["rationale"],
    "recommendations": high_priority["recommendations"],
    "ioc_summary": high_priority["ioc_summary"],
}